# Lab 04 — AI_CLASSIFY & AI_SENTIMENT

Two complementary functions for understanding text: **classify** assigns a category from a provided list; **sentiment** returns a score from -1 (negative) to +1 (positive).

| Function | Returns | Use Case |
|---|---|---|
| `AI_CLASSIFY` | Label + confidence score | Categorize orders, tickets, feedback |
| `AI_SENTIMENT` | Score -1 to +1 | Measure customer satisfaction at scale |

**What You'll Learn**

- `AI_CLASSIFY` with custom labels, multi-label, and few-shot examples
- Aggregating classification results
- `AI_SENTIMENT` — overall and aspect-based sentiment
- Star-rating vs AI sentiment mismatch detection

## Step 1 — Environment Setup

> **AI SQL Functions** — This lab uses the preferred `AI_` prefixed functions. 
Equivalent legacy functions: `AI_CLASSIFY` (replaces `SNOWFLAKE.CORTEX.CLASSIFY_TEXT`), `AI_SENTIMENT` (replaces `SNOWFLAKE.CORTEX.SENTIMENT / ENTITY_SENTIMENT`).

In [ ]:
USE ROLE DS_ROLE;
USE WAREHOUSE DS_WH;

-- Database pre-created by SYSADMIN in 00-admin-setup
USE DATABASE GENAI_LEARNING;
USE SCHEMA PUBLIC;

## Step 2 — AI_CLASSIFY: Categorize Order Comments

Classify free-text comments into business categories — no training needed.

In [ ]:
CREATE OR REPLACE VIEW orders_sample AS
SELECT
    o_orderkey AS order_key, o_orderstatus AS order_status,
    o_totalprice AS total_price, o_orderdate AS order_date,
    o_orderpriority AS order_priority, o_comment AS order_comment
FROM SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.ORDERS
LIMIT 500;

In [ ]:
SELECT
    order_key,
    order_comment,
    AI_CLASSIFY(
        order_comment,
        ['shipping_issue', 'pricing_concern', 'general_inquiry', 'urgent_request', 'positive_feedback']
    ) AS classification
FROM orders_sample
LIMIT 20;

In [ ]:
WITH classified AS (
    SELECT
        order_priority,
        AI_CLASSIFY(
            order_comment,
            ['shipping_issue', 'pricing_concern', 'general_inquiry', 'urgent_request', 'positive_feedback']
        ):labels[0]::STRING AS category
    FROM orders_sample
)
SELECT category, COUNT(*) AS cnt
FROM classified
GROUP BY category
ORDER BY cnt DESC;

## Step 2b — Advanced AI_CLASSIFY Options

AI_CLASSIFY supports several powerful options via the config object:

| Option | Purpose |
|---|---|
| `output_mode: 'multi'` | Multi-label classification (item can belong to multiple categories) |
| `task_description` | Context for the classification task (up to 50 words) |
| Label `description` | Per-category description to improve accuracy |
| `examples` | Few-shot learning with labeled examples |

In [ ]:
-- output_mode: 'multi' returns ALL matching labels per input
-- Using inline complaints that clearly span multiple categories
WITH complaints AS (
    SELECT 1 AS id, 'Package arrived 2 weeks late, completely smashed, and I was charged twice for shipping' AS comment
    UNION ALL SELECT 2, 'Great product but delivery was slow and customer service never responded to my ticket'
    UNION ALL SELECT 3, 'Wrong item sent and the return process is confusing, very disappointed'
    UNION ALL SELECT 4, 'Fast shipping but the item quality is poor and the price seems too high for what you get'
)
SELECT id,
    LEFT(comment, 70) AS comment_preview,
    AI_CLASSIFY(
        comment,
        ['shipping_issue', 'pricing_concern', 'product_quality', 'customer_service'],
        {'output_mode': 'multi'}
    ):labels AS categories
FROM complaints;

In [ ]:
SELECT
    order_key,
    order_comment,
    AI_CLASSIFY(
        order_comment,
        [
            {'label': 'shipping_issue',   'description': 'complaints about delivery delays or damaged packages'},
            {'label': 'pricing_concern',  'description': 'feedback about pricing, discounts, or billing errors'},
            {'label': 'general_inquiry',  'description': 'general questions or neutral comments'},
            {'label': 'product_quality',  'description': 'feedback about product defects or quality issues'}
        ],
        {
            'task_description': 'Classify order comments from an e-commerce platform into support categories',
            'output_mode': 'multi',
            'examples': [
                {
                    'input': 'The package arrived late and the box was crushed',
                    'labels': ['shipping_issue', 'product_quality'],
                    'explanation': 'Late delivery is a shipping issue, crushed box suggests product quality concern'
                }
            ]
        }
    ):labels AS categories
FROM orders_sample
LIMIT 5;

## Step 3 — AI_SENTIMENT: Product Review Scoring

Returns a continuous score from -1 (negative) to +1 (positive). Works on any text with no training.

In [ ]:
CREATE OR REPLACE TABLE product_reviews (
    review_id INT AUTOINCREMENT, product_name VARCHAR(200),
    category VARCHAR(100), review_text TEXT, star_rating FLOAT
);

INSERT INTO product_reviews (product_name, category, review_text, star_rating) VALUES
('Wireless Headphones Pro', 'Electronics', 'Absolutely love these headphones! Crystal clear sound and the noise cancellation is incredible.', 5.0),
('Wireless Headphones Pro', 'Electronics', 'Sound quality is decent but they hurt my ears after an hour. Padding needs improvement.', 3.0),
('Wireless Headphones Pro', 'Electronics', 'Terrible battery life. Dies after 2 hours despite claiming 20 hours. False advertising.', 1.0),
('Running Shoes Ultra', 'Sports', 'These shoes transformed my running experience. Super lightweight and great arch support.', 5.0),
('Running Shoes Ultra', 'Sports', 'Good shoes but the sizing runs small. Had to exchange for a larger pair.', 3.0),
('Running Shoes Ultra', 'Sports', 'Sole came apart after just two weeks of use. Very disappointing quality.', 1.0),
('Smart Home Hub', 'Electronics', 'Setup was a breeze and it works perfectly with all my devices. Voice control is very responsive.', 5.0),
('Smart Home Hub', 'Electronics', 'Works okay but the app is clunky and crashes frequently on Android.', 2.0),
('Smart Home Hub', 'Electronics', 'Constantly loses connection to WiFi. Had to restart it multiple times daily.', 1.0);

## Step 3b — Category-Based Sentiment (Aspect Analysis)

AI_SENTIMENT accepts an optional categories array for aspect-based analysis. Each category returns one of: `positive`, `negative`, `neutral`, `mixed`, or `unknown`.

In [ ]:
SELECT
    product_name,
    review_text,
    AI_SENTIMENT(
        review_text,
        ['product_quality', 'value_for_money', 'customer_service', 'ease_of_use']
    ) AS aspect_sentiment
FROM product_reviews
LIMIT 5;

In [ ]:
SELECT
    product_name,
    LEFT(review_text, 60) AS review_preview,
    cat.value:name::STRING AS aspect,
    cat.value:sentiment::STRING AS sentiment
FROM product_reviews,
    LATERAL FLATTEN(
        AI_SENTIMENT(
            review_text,
            ['product_quality', 'value_for_money', 'customer_service']
        ):categories
    ) AS cat
WHERE cat.value:name::STRING != 'overall'
LIMIT 15;

In [ ]:
SELECT
    product_name, star_rating,
    AI_SENTIMENT(review_text):categories[0]:sentiment::STRING AS sentiment,
    LEFT(review_text, 80) AS review_snippet
FROM product_reviews
ORDER BY star_rating DESC;

In [ ]:
SELECT
    product_name,
    COUNT(*) AS review_count,
    ROUND(AVG(star_rating), 2) AS avg_stars,
    COUNT_IF(AI_SENTIMENT(review_text):categories[0]:sentiment::STRING = 'positive') AS positive_count,
    COUNT_IF(AI_SENTIMENT(review_text):categories[0]:sentiment::STRING = 'negative') AS negative_count,
    COUNT_IF(AI_SENTIMENT(review_text):categories[0]:sentiment::STRING = 'neutral') AS neutral_count
FROM product_reviews
GROUP BY product_name
ORDER BY positive_count DESC;

## Step 4 — Find Star-Sentiment Mismatches

Detect reviews where the star rating doesn't match the text sentiment — potential gaming or errors.

In [ ]:
-- Flag reviews where star rating contradicts AI sentiment
WITH scored AS (
    SELECT
        product_name,
        star_rating,
        AI_SENTIMENT(review_text):categories[0]:sentiment::STRING AS sentiment,
        review_text
    FROM product_reviews
)
SELECT product_name, star_rating, sentiment, review_text
FROM scored
WHERE (star_rating >= 4 AND sentiment = 'negative')
   OR (star_rating <= 2 AND sentiment = 'positive')
ORDER BY star_rating DESC;

## Key Takeaways

- `AI_CLASSIFY` returns JSON with `label` and `score` — any business categories, no training.
- `AI_SENTIMENT` returns a continuous score (-1 to +1) — great for ranking and thresholds.
- Combine with `GROUP BY` for instant analytics dashboards.
- Compare AI sentiment vs. star ratings to detect gaming or mismatched feedback.
